# CineEmbed — EDA v2

Pipeline-first refresh of the EDA notebook. Applies all 13 fixes from the audit (see `docs/superpowers/specs/2026-05-03-eda-v2-design.md`). Produces a clean (329044, 564) feature matrix.

**Sections:**
- §1 Setup & Reproducibility
- §2 Pipeline Function Definitions
- §3 Pipeline Execution
- §4 EDA Visualizations
- §5 Persistence


In [1]:
# §1 — Setup & Reproducibility
import os, json, hashlib, random, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from scipy import stats

# Optional GPU stack — only imported if available
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CONFIG = {
    'seed': 42,
    'data_dir': Path('data'),
    'artifacts_dir': Path('artifacts'),
    'figures_dir': Path('artifacts/figures'),

    # File paths
    'paths': {
        'details':            Path('data/AllMoviesDetailsCleaned.csv'),
        'casting':            Path('data/AllMoviesCastingRaw.csv'),
        'awards':             Path('data/220k_awards_by_directors.csv'),
        'director_bios':      Path('data/500 favorite directors_with wikipedia summary.csv'),
        'director_langs':     Path('data/MostCommonLanguageByDirector.csv'),
        'lang_to_country':    Path('data/language to country.csv'),
    },

    # Feature engineering knobs (single source of truth — fix #11 + clean ablation)
    'top_n_genres': 20,
    'top_n_languages': 30,
    'q99_clip_threshold': 0.99,
    'runtime_clip': (10, 300),

    # Director profile (extension)
    'top_n_director_country': 15,
    'bio_pca_n_components':   64,
    'bio_embedding_cache':    Path('artifacts/director_bio_embeddings.npy'),
    'bio_embedding_meta':     Path('artifacts/director_bio_embeddings.meta.json'),
    'bio_pca_cache':          Path('artifacts/director_bio_pca.pkl'),
    'director_profile_meta':  Path('artifacts/director_profile_metadata.json'),

    # Embedding model (fix #5 — multilingual)
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
    'embedding_batch_size': 64,
    'embedding_dim': 384,
    'embedding_cache': Path('artifacts/text_embeddings.npy'),
    'embedding_meta': Path('artifacts/text_embeddings.meta.json'),
}

CONFIG['artifacts_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG['seed'])

# Verify new data files exist
for k in ('director_bios', 'director_langs', 'lang_to_country'):
    assert CONFIG['paths'][k].exists(), f"missing data file: {CONFIG['paths'][k]}"

# Reproducibility self-check
_check = np.random.rand(3)
print("\u2705 \u00a71 Setup complete")
print(f"   seed = {CONFIG['seed']}")
print(f"   np.random sample (deterministic) = {_check}")
print(f"   torch available = {HAS_TORCH}")


ModuleNotFoundError: No module named 'seaborn'

## §2.1 — Data Layer
Pure functions: `load_csvs`, `normalize_director_name`, `merge_details_casting`. Each function has a sanity-test cell immediately after.

In [ ]:
# §2.1 — Data layer functions
import unicodedata

def normalize_director_name(name: str | None) -> str:
    """Stable director key for joins.

    Steps:
      1. None / NaN / empty → ''
      2. Unicode NFKD decompose, strip combining marks (accents)
      3. Swap "Last, First" → "First Last"
      4. Lowercase, collapse internal whitespace, strip ends.
    """
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ''
    s = str(name).strip()
    if not s:
        return ''
    # NFKD + ascii filter (drops accents)
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    # "Last, First" → "First Last"
    if ',' in s:
        parts = [p.strip() for p in s.split(',', 1)]
        if len(parts) == 2 and parts[0] and parts[1]:
            s = f"{parts[1]} {parts[0]}"
    # collapse whitespace, lowercase
    s = ' '.join(s.split()).lower()
    return s


In [ ]:
# Test §2.1: normalize_director_name (fix #8)
_cases = [
    ('Steven Spielberg',   'steven spielberg'),
    ('Spielberg, Steven',  'steven spielberg'),
    ('  Pedro  Almodóvar ', 'pedro almodovar'),
    ('Léa  Pool',          'lea pool'),
    (None,                 ''),
    ('',                   ''),
    ('SCORSESE, MARTIN',   'martin scorsese'),
]
for raw, expected in _cases:
    got = normalize_director_name(raw)
    assert got == expected, f"normalize_director_name({raw!r}) = {got!r} != {expected!r}"
print(f"\u2705 normalize_director_name: {len(_cases)} cases pass")


In [ ]:
def load_csvs(paths: dict[str, Path]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the three source CSVs with their correct separators.

    details, casting use ';' (TMDB-derived); awards uses ','.
    """
    details = pd.read_csv(paths['details'], sep=';', low_memory=False)
    casting = pd.read_csv(paths['casting'], sep=';', low_memory=False)
    awards  = pd.read_csv(paths['awards'],  sep=',', low_memory=False)
    return details, casting, awards


In [ ]:
def merge_details_casting(details: pd.DataFrame, casting: pd.DataFrame) -> pd.DataFrame:
    """Inner-shape merge on `id`. Adds `director_name_norm` (fix #8) for award joins.

    Carries `director_name` (raw) for human readability and `director_name_norm`
    for joins. Only the necessary casting columns are pulled in to avoid bloat.
    """
    casting_slim = casting[['id', 'director_name', 'director_gender']].copy()
    casting_slim['director_name_norm'] = casting_slim['director_name'].apply(normalize_director_name)
    merged = details.merge(casting_slim, on='id', how='left')
    return merged


In [ ]:
# Test §2.1: merge_details_casting
_details = pd.DataFrame({'id': [1, 2, 3], 'title': ['A', 'B', 'C']})
_casting = pd.DataFrame({
    'id': [1, 2, 3],
    'director_name': ['Steven Spielberg', 'Spielberg, Steven', None],
    'director_gender': [2, 2, 0],
})
_merged = merge_details_casting(_details, _casting)
assert set(_merged.columns) >= {'id', 'title', 'director_name', 'director_name_norm', 'director_gender'}
assert _merged.loc[0, 'director_name_norm'] == 'steven spielberg'
assert _merged.loc[1, 'director_name_norm'] == 'steven spielberg'   # normalized form merges
assert _merged.loc[2, 'director_name_norm'] == ''
print("✅ merge_details_casting passes")


### §2.1 Director Profile CSVs

In [ ]:
def load_director_profile_csvs(paths: dict) -> tuple:
    """Load the three director-profile CSVs with their idiosyncratic formats.

    Args:
        paths: dict with keys 'director_bios', 'director_langs', 'lang_to_country'.

    Returns:
        (bios_df, langs_df, lang_to_country_df). Director names are leading/trailing
        whitespace-stripped. Empty bios are dropped from bios_df.
    """
    # 1. Bios — no header, semicolon-separated, two columns
    bios = pd.read_csv(
        paths['director_bios'],
        sep=';', header=None,
        names=['director_name', 'bio'],
        dtype=str, encoding='utf-8',
    )
    bios['director_name'] = bios['director_name'].str.strip()
    bios['bio'] = bios['bio'].fillna('').str.strip()
    bios = bios[bios['bio'].str.len() > 0].reset_index(drop=True)

    # 2. Langs — header, comma-separated; some director names have a leading TAB
    langs = pd.read_csv(paths['director_langs'], sep=',', dtype={'nb': int})
    langs['director_name'] = langs['director_name'].str.strip()
    langs = langs.dropna(subset=['director_name', 'original_language']).reset_index(drop=True)

    # 3. Lang-to-country — header, semicolon-separated; lower-case columns for consistency
    l2c = pd.read_csv(paths['lang_to_country'], sep=';')
    l2c.columns = [c.strip().lower() for c in l2c.columns]
    l2c['language'] = l2c['language'].str.strip()
    l2c['country']  = l2c['country'].str.strip()

    return bios, langs, l2c


In [ ]:
# Synthetic mini-CSVs in a tmp dir to assert load behavior
import tempfile, shutil
_tmp = Path(tempfile.mkdtemp())

(_tmp / 'bios.csv').write_text(
    'Steven Spielberg;is an american filmmaker known for jaws and e.t.\n'
    'Aki Kaurism\u00e4ki;is a finnish film director.\n'
    'Empty Director;\n',
    encoding='utf-8',
)
(_tmp / 'langs.csv').write_text(
    'director_name,original_language,nb\n'
    '\tSteven Spielberg,en,30\n'
    'Steven Spielberg,fr,1\n'
    'Aki Kaurism\u00e4ki,fi,15\n',
    encoding='utf-8',
)
(_tmp / 'l2c.csv').write_text(
    'Language;Country\nen;USA\nfi;FIN\nfr;FRA\n',
    encoding='utf-8',
)

bios, langs, l2c = load_director_profile_csvs({
    'director_bios':   _tmp / 'bios.csv',
    'director_langs':  _tmp / 'langs.csv',
    'lang_to_country': _tmp / 'l2c.csv',
})

# 1. Bios: 3 rows in raw, 2 should remain after dropping empty bios
assert list(bios.columns) == ['director_name', 'bio'], bios.columns.tolist()
assert len(bios) == 2, f"expected 2 non-empty bios, got {len(bios)}"
assert 'Empty Director' not in bios['director_name'].values

# 2. Langs: leading tab stripped from director_name
assert (langs['director_name'] == '\tSteven Spielberg').sum() == 0
assert (langs['director_name'] == 'Steven Spielberg').sum() == 2

# 3. lang_to_country: 3 rows, columns lower-cased
assert list(l2c.columns) == ['language', 'country']
assert len(l2c) == 3

shutil.rmtree(_tmp)
print("\u2705 load_director_profile_csvs: all 3 assertions passed")


In [ ]:
def normalize_director_in_profile_dfs(bios_df: pd.DataFrame, langs_df: pd.DataFrame
) -> tuple:
    """Add a `director_name_norm` column to both dataframes using the existing
    normalize_director_name (defined in parent §2.1).

    Returns:
        (bios_df_with_norm, langs_df_with_norm) — copies, originals untouched.
    """
    bios_out  = bios_df.copy()
    langs_out = langs_df.copy()
    bios_out['director_name_norm']  = bios_out['director_name'].map(normalize_director_name)
    langs_out['director_name_norm'] = langs_out['director_name'].map(normalize_director_name)
    return bios_out, langs_out


In [ ]:
# Test: applies the existing normalize_director_name to add a director_name_norm column
bios = pd.DataFrame({'director_name': ['Aki Kaurism\u00e4ki', 'Spielberg, Steven', 'Wes  Anderson']})
langs = pd.DataFrame({
    'director_name': ['Aki Kaurism\u00e4ki', 'Steven Spielberg'],
    'original_language': ['fi', 'en'],
    'nb': [15, 30],
})

bios_n, langs_n = normalize_director_in_profile_dfs(bios, langs)

# director_name_norm column added
assert 'director_name_norm' in bios_n.columns
assert 'director_name_norm' in langs_n.columns

# Same director resolves to the same normalized form across both dfs
spielberg_in_bios  = bios_n.loc[bios_n['director_name'] == 'Spielberg, Steven',  'director_name_norm'].iloc[0]
spielberg_in_langs = langs_n.loc[langs_n['director_name'] == 'Steven Spielberg', 'director_name_norm'].iloc[0]
assert spielberg_in_bios == spielberg_in_langs, f"{spielberg_in_bios!r} != {spielberg_in_langs!r}"

# Accents stripped, casefolded, double-space collapsed
kaurismaki_norm = bios_n.loc[bios_n['director_name'] == 'Aki Kaurism\u00e4ki', 'director_name_norm'].iloc[0]
assert '\u00e4' not in kaurismaki_norm and kaurismaki_norm == kaurismaki_norm.lower()
wes_norm = bios_n.loc[bios_n['director_name'] == 'Wes  Anderson', 'director_name_norm'].iloc[0]
assert '  ' not in wes_norm

print("\u2705 normalize_director_in_profile_dfs: all 4 assertions passed")


## §2.2 — Awards Layer
Temporal-aware per-film aggregation. Implements fixes #1 (temporal leak) and #9 (Oscar/Palme regex word-boundary).

In [ ]:
import re

# OSCAR_RE: Word-boundary match for "Oscar" with a negative lookahead that
# excludes "Oscar Wilde Award" (Oscar Wilde is a person, not the Oscar award).
# The plan's audit identified this as a real-world false-positive trap.
OSCAR_RE = re.compile(r'\bOscar\b(?!\s+Wilde)', flags=re.IGNORECASE)
PALME_RE = re.compile(r'\bPalme\b', flags=re.IGNORECASE)


def aggregate_awards_temporal(
    awards: pd.DataFrame,
    films: pd.DataFrame,
) -> pd.DataFrame:
    """Per-film aggregation of director awards.

    Two fixes:
      - #1 (temporal): only awards with year <= film.release_year contribute.
      - #9 (regex):    Oscar/Palme detection uses \\b word-boundary.

    Returns a DataFrame with one row per film id and columns:
      prior_total_nominations, prior_total_wins,
      prior_oscar_nominations, prior_oscar_wins,
      prior_palme_nominations, prior_palme_wins

    Note: *_nominations counts entries where the director was nominated but did
    NOT win (i.e., is_oscar/is_palme AND NOT is_won). *_wins counts entries
    where they won (is_oscar/is_palme AND is_won).
    """
    # Normalize director key on awards side
    awards = awards.copy()
    awards['director_name_norm'] = awards['director_name'].apply(normalize_director_name)

    # Year column (if missing, parse from a date-like field; here we trust 'year')
    awards['year'] = pd.to_numeric(awards['year'], errors='coerce')
    awards = awards.dropna(subset=['year'])
    awards['year'] = awards['year'].astype(int)

    # Annotate flags up-front (avoid per-row regex during the join)
    awards['is_won']   = (awards['outcome'].fillna('') == 'Won')
    awards['is_oscar'] = awards['category'].fillna('').str.contains(OSCAR_RE, regex=True)
    awards['is_palme'] = awards['category'].fillna('').str.contains(PALME_RE, regex=True)

    # Films side — derive year
    films = films.copy()
    films['release_year'] = pd.to_datetime(films['release_date'], errors='coerce').dt.year

    # For each (director_norm), pre-sort awards ascending by year for cumulative aggregation
    award_groups = awards.sort_values('year').groupby('director_name_norm')

    rows = []
    for _, film in films[['id', 'director_name_norm', 'release_year']].iterrows():
        out = {
            'id': film['id'],
            'prior_total_nominations': 0,
            'prior_total_wins': 0,
            'prior_oscar_nominations': 0,
            'prior_oscar_wins': 0,
            'prior_palme_nominations': 0,
            'prior_palme_wins': 0,
        }
        director = film['director_name_norm']
        year = film['release_year']
        if not director or pd.isna(year):
            rows.append(out)
            continue
        if director not in award_groups.groups:
            rows.append(out)
            continue
        sub = award_groups.get_group(director)
        sub = sub[sub['year'] <= year]
        if sub.empty:
            rows.append(out)
            continue
        out['prior_total_nominations']  = int((~sub['is_won']).sum())
        out['prior_total_wins']         = int(sub['is_won'].sum())
        out['prior_oscar_nominations']  = int((sub['is_oscar'] & ~sub['is_won']).sum())
        out['prior_oscar_wins']         = int((sub['is_oscar'] &  sub['is_won']).sum())
        out['prior_palme_nominations']  = int((sub['is_palme'] & ~sub['is_won']).sum())
        out['prior_palme_wins']         = int((sub['is_palme'] &  sub['is_won']).sum())
        rows.append(out)

    return pd.DataFrame(rows)


In [ ]:
# Test §2.2: aggregate_awards_temporal
# Synthetic awards: director "alice" has 2 wins (1995, 2010), 1 nom (2005)
# Director "bob" has 1 oscar win (2000) and 1 "Oscar Wilde Award" nom (1998 — false positive trap)
_awards = pd.DataFrame({
    'director_name': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob'],
    'category':      ['Best Director', 'Best Picture', 'Best Director — Oscar',
                      'Academy Award (Oscar)', 'Oscar Wilde Award'],
    'outcome':       ['Won', 'Nominated', 'Won', 'Won', 'Nominated'],
    'year':          [1995, 2005, 2010, 2000, 1998],
})
_films = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'director_name_norm': ['alice', 'alice', 'bob', 'bob'],
    'release_date': pd.to_datetime(['1990-01-01', '2008-01-01', '1995-01-01', '2005-01-01']),
})
_agg = aggregate_awards_temporal(_awards, _films)

# Assertions (one per fix):
# Fix #1 — temporal: alice 1990 film sees 0 wins; alice 2008 film sees 1 win (1995)
row_a1990 = _agg.set_index('id').loc[101]
row_a2008 = _agg.set_index('id').loc[102]
assert row_a1990['prior_total_wins'] == 0,  f"1990 leak: {row_a1990['prior_total_wins']}"
assert row_a2008['prior_total_wins'] == 1,  f"2008 expects 1 win, got {row_a2008['prior_total_wins']}"

# Fix #9 — word-boundary: bob 1995 sees 0 oscar wins (Wilde shouldn't count, win is 2000)
# bob 2005 sees 1 oscar win (2000) and 0 oscar noms (Wilde is filtered out)
row_b1995 = _agg.set_index('id').loc[103]
row_b2005 = _agg.set_index('id').loc[104]
assert row_b1995['prior_oscar_wins'] == 0
assert row_b2005['prior_oscar_wins'] == 1
assert row_b2005['prior_oscar_nominations'] == 0, \
    f"Wilde should not count, got {row_b2005['prior_oscar_nominations']}"

print("\u2705 aggregate_awards_temporal: temporal cutoff + regex word-boundary work")


## §2.3 — Feature Engineering
Five engineer_* functions, one per modality. Each returns a DataFrame block aligned by row index with the master frame.

### §2.3.a — Numerical (fixes #2, #7, #10)

In [ ]:
def engineer_numerical(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """Numerical block — 6 columns.

    Fixes:
      #2 vote_average imputation — use mean of voted-only films, NOT median (which is 0)
      #7 vote_count Q99 clip + log (was: log only)
      #10 has_engagement flag
    """
    out = pd.DataFrame(index=df.index)

    pop = pd.to_numeric(df['popularity'], errors='coerce').fillna(0)
    vc  = pd.to_numeric(df['vote_count'], errors='coerce').fillna(0)
    rt  = pd.to_numeric(df['runtime'], errors='coerce')
    va  = pd.to_numeric(df['vote_average'], errors='coerce')

    # Q99 clip then log1p
    pop_q99 = pop.quantile(config['q99_clip_threshold'])
    vc_q99  = vc.quantile(config['q99_clip_threshold'])
    out['log_popularity'] = np.log1p(pop.clip(upper=pop_q99).clip(lower=0))
    out['log_vote_count'] = np.log1p(vc.clip(upper=vc_q99).clip(lower=0))

    # runtime: cap to [10, 300] then min-max
    rt_lo, rt_hi = config['runtime_clip']
    rt_filled = rt.fillna(rt.median() if rt.notna().any() else (rt_lo + rt_hi) / 2)
    rt_capped = rt_filled.clip(lower=rt_lo, upper=rt_hi)
    out['runtime_norm'] = (rt_capped - rt_lo) / (rt_hi - rt_lo)

    # vote_average — fix #2 (smart imputation)
    voted_mask = vc > 0
    if voted_mask.any():
        imputed_value = va[voted_mask].mean()  # ~6.0 on real data
    else:
        imputed_value = 0.0
    va_filled = va.where(voted_mask & va.notna(), imputed_value)
    # min-max on [0, 10] natural scale
    out['vote_average_norm'] = (va_filled.clip(0, 10) / 10.0)

    # Flags — fixes #2, #10
    out['has_vote']        = voted_mask.astype(np.int8)
    out['has_engagement']  = ((pop > 0) | (vc > 0)).astype(np.int8)

    return out

In [ ]:
# Test §2.3.a: engineer_numerical
_df = pd.DataFrame({
    'popularity':   [0.0, 0.5, 1.0, 100.0, np.nan, 0.0],
    'vote_count':   [0,   10,  50,  10000, 5,      0],
    'runtime':      [120, 95,  np.nan, 5,   500,   90],
    'vote_average': [np.nan, 7.5, 8.0, 6.5, 5.0, np.nan],
})
_out = engineer_numerical(_df, CONFIG)

# fix #2: vote_average imputation
# Voted-only mean = mean of [7.5, 8.0, 6.5, 5.0] = 6.75
# Films with vote_count == 0 (rows 0 and 5) → vote_average filled with ~6.75
assert abs(_out.loc[0, 'vote_average_norm'] - _out.loc[5, 'vote_average_norm']) < 1e-9
# Both should NOT be the minimum (which would happen with median=0 imputation)
assert _out['vote_average_norm'].min() < _out.loc[0, 'vote_average_norm'] or \
       _out['vote_average_norm'].max() > _out.loc[0, 'vote_average_norm']

# fix #7: vote_count Q99 clip applied — log_vote_count for 10000 should not blow up
assert _out['log_vote_count'].max() < np.log1p(10001)

# fix #10: has_engagement flag
assert _out.loc[0, 'has_engagement'] == 0  # popularity=0 AND vote_count=0
assert _out.loc[1, 'has_engagement'] == 1  # vote_count=10 > 0
assert _out.loc[5, 'has_engagement'] == 0

# has_vote flag
assert _out.loc[0, 'has_vote'] == 0
assert _out.loc[1, 'has_vote'] == 1

# All 6 columns present, no NaN
expected_cols = {'log_popularity', 'log_vote_count', 'runtime_norm',
                 'vote_average_norm', 'has_vote', 'has_engagement'}
assert set(_out.columns) == expected_cols
assert _out.isna().sum().sum() == 0

print(f"✅ engineer_numerical: 6-col output, {len(_out)} rows, no NaN")

### §2.3.b — Genre & Language (fix #3 — Unknown genre + has_genre flag)

In [ ]:
def engineer_genres(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Genre block — top-N multi-hot + Unknown + has_genre flag (fix #3).

    Empty/None genre lists become ['Unknown'] AND has_genre = 0.
    """
    raw = df['genres'].fillna('').astype(str)
    genres_list = raw.apply(
        lambda x: [g.strip() for g in x.split('|') if g.strip()]
    )
    has_genre = genres_list.apply(lambda lst: 1 if lst else 0).astype(np.int8)

    # Pick top-N from non-empty rows
    counts = {}
    for lst in genres_list:
        for g in lst:
            counts[g] = counts.get(g, 0) + 1
    top_genres = [g for g, _ in sorted(
        counts.items(),
        key=lambda kv: (-kv[1], kv[0]),
    )[:top_n]]

    # Replace empty with ['Unknown'] for encoding
    genres_for_mlb = genres_list.apply(lambda lst: lst if lst else ['Unknown'])

    # Restrict to top_genres + 'Unknown'
    keep = set(top_genres) | {'Unknown'}
    genres_filtered = genres_for_mlb.apply(lambda lst: [g for g in lst if g in keep] or ['Unknown'])

    mlb = MultiLabelBinarizer(classes=sorted(keep))
    encoded = mlb.fit_transform(genres_filtered)
    out = pd.DataFrame(
        encoded,
        columns=[f'genre_{g}' for g in mlb.classes_],
        index=df.index,
        dtype=np.int8,
    )
    out['has_genre'] = has_genre.values
    return out

In [ ]:
# Test §2.3.b: engineer_genres (fix #3)
_df = pd.DataFrame({
    'genres': ['Drama|Comedy', 'Drama', None, '', 'Action|Thriller', 'Drama|Comedy'],
})
_out = engineer_genres(_df, top_n=3)

# Top 3 by frequency: Drama (3), Comedy (2), Action (1) (or Thriller — tiebreak by alpha)
# So expected columns: genre_Drama, genre_Comedy, genre_Action OR Thriller, genre_Unknown, has_genre
expected_genre_cols = {'genre_Drama', 'genre_Comedy', 'genre_Unknown', 'has_genre'}
assert expected_genre_cols.issubset(set(_out.columns)), f"missing cols. got: {set(_out.columns)}"

# Fix #3: empty/None → genre_Unknown=1, has_genre=0
assert _out.loc[2, 'genre_Unknown'] == 1
assert _out.loc[3, 'genre_Unknown'] == 1
assert _out.loc[2, 'has_genre'] == 0
assert _out.loc[3, 'has_genre'] == 0

# Non-empty rows: has_genre = 1, genre_Unknown = 0
assert _out.loc[0, 'has_genre'] == 1
assert _out.loc[0, 'genre_Unknown'] == 0
assert _out.loc[0, 'genre_Drama'] == 1
assert _out.loc[0, 'genre_Comedy'] == 1

print(f"✅ engineer_genres: {len(_out.columns)} cols, fix #3 holds")

### §2.3.c — Language

In [ ]:
def engineer_languages(df: pd.DataFrame, top_n: int = 30) -> pd.DataFrame:
    """Language block — top-N + 'other' one-hot."""
    raw = df['original_language'].fillna('other').astype(str)
    counts = raw.value_counts()
    top_langs = counts.head(top_n).index.tolist()
    grouped = raw.where(raw.isin(top_langs), other='other')
    out = pd.get_dummies(grouped, prefix='lang', dtype=np.int8)
    # Ensure 'lang_other' column exists even if no rows fell out of top-N
    if 'lang_other' not in out.columns:
        out['lang_other'] = np.int8(0)
    return out

In [ ]:
# Test §2.3.b: engineer_languages
_df = pd.DataFrame({
    'original_language': ['en', 'en', 'fr', 'tr', 'jp', 'unknown_lang_xyz', None],
})
_out = engineer_languages(_df, top_n=3)
# Top 3: en (2), fr (1), jp/tr (1) — alpha tiebreak
expected_subset = {'lang_en', 'lang_fr', 'lang_other'}
assert expected_subset.issubset(set(_out.columns))
assert _out.loc[5, 'lang_other'] == 1   # unknown_lang_xyz
assert _out.loc[6, 'lang_other'] == 1   # None

# Each row has exactly one '1' across all lang columns (one-hot)
assert (_out.sum(axis=1) == 1).all()
print(f"✅ engineer_languages: one-hot, {len(_out.columns)} cols")

### §2.3.d — Decade (fix #6)

In [ ]:
def engineer_decade(df: pd.DataFrame) -> pd.DataFrame:
    """Decade block — fix #6.

    For films with parseable release_date:
      decade_norm = (decade - 1900) / 130, where decade = year // 10 * 10.
    For films without:
      decade_norm = 0.0 AND has_release_date = 0 — model can learn to ignore decade.
    """
    rd = pd.to_datetime(df['release_date'], errors='coerce')
    has = rd.notna()
    year = rd.dt.year.where(has, other=np.nan)
    decade = (year // 10 * 10).where(has, other=np.nan)
    decade_norm = ((decade - 1900) / 130).where(has, other=0.0)

    out = pd.DataFrame({
        'decade_norm': decade_norm.astype(float).values,
        'has_release_date': has.astype(np.int8).values,
    }, index=df.index)
    return out

In [ ]:
# Test §2.3.c: engineer_decade (fix #6)
_df = pd.DataFrame({
    'release_date': ['1990-05-01', '2020-01-01', '1900-01-01', None, 'invalid'],
})
_df['release_date'] = pd.to_datetime(_df['release_date'], errors='coerce')
_out = engineer_decade(_df)

assert set(_out.columns) == {'decade_norm', 'has_release_date'}
# (1990 - 1900) / 130 = 0.6923...
assert abs(_out.loc[0, 'decade_norm'] - (90/130)) < 1e-9
# (2020 - 1900) / 130 = 0.923...
assert abs(_out.loc[1, 'decade_norm'] - (120/130)) < 1e-9
# 1900 → 0.0
assert _out.loc[2, 'decade_norm'] == 0.0
# Missing/invalid → decade_norm = 0.0 AND has_release_date = 0
assert _out.loc[3, 'decade_norm'] == 0.0
assert _out.loc[3, 'has_release_date'] == 0
assert _out.loc[4, 'decade_norm'] == 0.0
assert _out.loc[4, 'has_release_date'] == 0
# Valid rows → has_release_date = 1
assert _out.loc[0, 'has_release_date'] == 1
assert _out.loc[2, 'has_release_date'] == 1

print("✅ engineer_decade: fix #6 (normalized + has_release_date) holds")

### §2.3.e — Director Awards

In [ ]:
AWARDS_RAW_COLS = [
    'prior_total_nominations', 'prior_total_wins',
    'prior_oscar_nominations', 'prior_oscar_wins',
    'prior_palme_nominations', 'prior_palme_wins',
]

def engineer_director_awards(df: pd.DataFrame) -> pd.DataFrame:
    """Director awards block — log1p of the 6 prior_ columns from §2.2."""
    out = pd.DataFrame(index=df.index)
    for col in AWARDS_RAW_COLS:
        clean = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)
        out[f'prior_log_{col[len("prior_"):]}'] = np.log1p(clean)
    return out

In [ ]:
# Test §2.3.c: engineer_director_awards
_df = pd.DataFrame({
    'prior_total_nominations': [0, 5, 100],
    'prior_total_wins': [0, 2, 50],
    'prior_oscar_nominations': [0, 1, 10],
    'prior_oscar_wins': [0, 0, 3],
    'prior_palme_nominations': [0, 0, 1],
    'prior_palme_wins': [0, 0, 1],
})
_out = engineer_director_awards(_df)
# All log1p applied
assert abs(_out.loc[0, 'prior_log_total_wins']) < 1e-9
assert abs(_out.loc[2, 'prior_log_total_wins'] - np.log1p(50)) < 1e-9
# 6 columns expected
assert len(_out.columns) == 6
# All non-negative
assert (_out.values >= 0).all()
print(f"✅ engineer_director_awards: 6 cols, log1p applied")

## §2.4 — Text Embedding (fixes #5, #12)
Multilingual sentence-transformers model with MD5-keyed file cache.

In [ ]:
def compute_text_embeddings(
    overviews: pd.Series,
    *,
    model_name: str,
    cache_path: Path,
    cache_meta_path: Path | None = None,
    batch_size: int = 64,
) -> np.ndarray:
    """Sentence embeddings with MD5-keyed file cache.

    Behaviour:
      • Cache hit if model_name + len(overviews) + sum(char_lengths) match.
      • Empty overviews produce a zero vector (preserves old behaviour, info captured by has_overview flag elsewhere).
      • On cache miss, encodes ALL overviews (zeros included) for simplicity, then overwrites empty rows with zeros.

    The first cold run on 329k films takes ~30-60 min on Colab T4 GPU; warm load is < 5 s.
    """
    cache_path = Path(cache_path)
    if cache_meta_path is None:
        cache_meta_path = cache_path.with_suffix('.meta.json')

    overviews = overviews.fillna('').astype(str)
    total_chars = int(overviews.str.len().sum())
    expected_hash = hashlib.md5(
        f"{model_name}|{len(overviews)}|{total_chars}".encode()
    ).hexdigest()

    # Cache hit?
    if cache_path.exists() and cache_meta_path.exists():
        try:
            meta = json.loads(cache_meta_path.read_text())
            if meta.get('hash') == expected_hash:
                emb = np.load(cache_path)
                if emb.shape == (len(overviews), 384):
                    return emb
        except Exception:
            pass  # fall through to cold path

    # Cold path — set USE_TF=0 to prevent TensorFlow from loading (avoids deadlock on macOS)
    os.environ.setdefault('USE_TF', '0')
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    texts = overviews.tolist()
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype(np.float32)

    # Force zero vectors for empty overviews
    empty_mask = (overviews.str.len() == 0).values
    emb[empty_mask] = 0.0

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, emb)
    cache_meta_path.write_text(json.dumps({
        'hash': expected_hash,
        'model': model_name,
        'n': len(overviews),
        'total_chars': total_chars,
        'created_utc': datetime.now(timezone.utc).isoformat(),
    }, indent=2))

    return emb


In [ ]:
# Test §2.4: compute_text_embeddings cache
import tempfile
import time

with tempfile.TemporaryDirectory() as _tmp:
    _cache = Path(_tmp) / 'emb.npy'
    _meta  = Path(_tmp) / 'emb.meta.json'
    _ovs = pd.Series(['A short film', '', 'Another plot'])

    # Cold path: compute, save
    _t_cold_start = time.time()
    _emb1 = compute_text_embeddings(
        _ovs,
        model_name=CONFIG['embedding_model'],
        cache_path=_cache,
        cache_meta_path=_meta,
        batch_size=4,
    )
    _t_cold = time.time() - _t_cold_start
    assert _emb1.shape == (3, 384), f'got {_emb1.shape}'
    assert _cache.exists()
    assert _meta.exists()

    # Empty overview must produce zero vector
    assert np.allclose(_emb1[1], 0.0), 'empty overview should be zero vector'
    assert not np.allclose(_emb1[0], 0.0), 'non-empty overview should be non-zero'

    # Warm path: should load from cache (verify by mtime — no recompute)
    _t0 = _cache.stat().st_mtime
    _t_warm_start = time.time()
    _emb2 = compute_text_embeddings(
        _ovs,
        model_name=CONFIG['embedding_model'],
        cache_path=_cache,
        cache_meta_path=_meta,
        batch_size=4,
    )
    _t_warm = time.time() - _t_warm_start
    assert np.array_equal(_emb1, _emb2), 'warm load should equal cold result'
    assert _cache.stat().st_mtime == _t0, 'cache should not have been rewritten'

    print(f'  cold: {_t_cold:.2f}s | warm: {_t_warm:.4f}s')
    print(f'  shape: {_emb1.shape} | zero-row verified | warm faster: {_t_warm < _t_cold}')

# tempfile.TemporaryDirectory cleans up automatically — no leftover test cache files
print('✅ compute_text_embeddings: cold path, warm path, zero-vector all work')


In [ ]:
def compute_director_bio_embeddings(
    bios_df,
    *,
    model_name: str,
    cache_path,
    meta_path,
    batch_size: int = 64,
) -> 'np.ndarray':
    """Embed director Wikipedia bios with a multilingual sentence-transformer.

    Same pattern as parent's `compute_text_embeddings`: cache key is MD5 of
    (model_name, n_directors, total_bio_chars). On cache hit the .npy is loaded
    and the model is skipped entirely.

    Args:
        bios_df: must have a 'bio' column; row order defines embedding order.

    Returns:
        ndarray of shape (len(bios_df), 384).
    """
    cache_path = Path(cache_path)
    meta_path  = Path(meta_path)

    # 1. Compute cache key from inputs
    n = len(bios_df)
    total_chars = int(bios_df['bio'].str.len().sum())
    key_str = f"{model_name}|{n}|{total_chars}"
    expected_md5 = hashlib.md5(key_str.encode('utf-8')).hexdigest()

    # 2. Cache hit?
    if cache_path.exists() and meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text())
            if meta.get('md5') == expected_md5 and meta.get('shape', [None])[0] == n:
                emb = np.load(cache_path)
                print(f"   [cache HIT] director bio embeddings: {emb.shape} loaded from {cache_path.name}")
                return emb
        except Exception:
            pass  # fall through to cold path

    # 3. Cold path — set USE_TF=0 to prevent TensorFlow deadlock (mirrors compute_text_embeddings)
    os.environ.setdefault('USE_TF', '0')
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    from sentence_transformers import SentenceTransformer
    print(f"   [cache MISS] computing {n} director bio embeddings (model={model_name})...")
    model = SentenceTransformer(model_name)
    emb = model.encode(
        bios_df['bio'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    # 4. Save cache
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, emb)
    meta_path.write_text(json.dumps({
        'md5': expected_md5,
        'shape': list(emb.shape),
        'model_name': model_name,
        'n_directors': n,
        'total_chars': total_chars,
        'created_at': datetime.now(timezone.utc).isoformat(),
    }, indent=2))

    return emb


In [ ]:
# Test §2.4: compute_director_bio_embeddings — 5 mini-bios, cold + warm call
import time

_tmp_cache = Path('artifacts/_test_bio_emb.npy')
_tmp_meta  = Path('artifacts/_test_bio_emb.meta.json')
for p in (_tmp_cache, _tmp_meta):
    if p.exists():
        p.unlink()

mini_bios = pd.DataFrame({
    'director_name': [f'Director_{i}' for i in range(5)],
    'bio': [
        'is an american filmmaker known for action movies.',
        'is a french auteur known for slow-paced cinema.',
        'is a japanese animator famous for studio ghibli.',
        'is a british director of art-house thrillers.',
        'is a turkish-german filmmaker working in europe.',
    ],
})

t0 = time.time()
emb_cold = compute_director_bio_embeddings(
    mini_bios, model_name=CONFIG['embedding_model'],
    cache_path=_tmp_cache, meta_path=_tmp_meta,
    batch_size=8,
)
cold_dt = time.time() - t0

assert emb_cold.shape == (5, 384), emb_cold.shape
assert _tmp_cache.exists() and _tmp_meta.exists()

# Warm call — same inputs, should hit cache and skip the model
t0 = time.time()
emb_warm = compute_director_bio_embeddings(
    mini_bios, model_name=CONFIG['embedding_model'],
    cache_path=_tmp_cache, meta_path=_tmp_meta,
    batch_size=8,
)
warm_dt = time.time() - t0

assert np.allclose(emb_cold, emb_warm), 'cache hit returned different embeddings'
assert warm_dt < cold_dt, f'warm not faster than cold: cold={cold_dt:.2f}s warm={warm_dt:.2f}s'

# Cleanup
_tmp_cache.unlink(); _tmp_meta.unlink()
print(f'\u2705 compute_director_bio_embeddings: cold={cold_dt:.1f}s, warm={warm_dt:.3f}s, shapes match')


In [ ]:
def pca_reduce_director_bios(
    emb: 'np.ndarray',
    n_components: int = 64,
    random_state: int = 42,
) -> tuple:
    """Reduce director-bio embeddings via PCA.

    The fitted PCA object is returned alongside the reduced array so it can be
    persisted to disk (re-applying it to held-out directors at inference time).

    Returns:
        (reduced_array of shape (n_dirs, n_components), fitted PCA instance).
    """
    pca = PCA(n_components=n_components, random_state=random_state)
    reduced = pca.fit_transform(emb).astype(np.float32)
    return reduced, pca


In [ ]:
# Test §2.4: pca_reduce_director_bios — reduce a (20, 384) synthetic embedding to 8 PCs
np.random.seed(0)
fake_emb = np.random.randn(20, 384).astype(np.float32)

reduced, pca_obj = pca_reduce_director_bios(fake_emb, n_components=8, random_state=42)

assert reduced.shape == (20, 8), reduced.shape
assert hasattr(pca_obj, 'explained_variance_ratio_')
assert len(pca_obj.explained_variance_ratio_) == 8
# Random data → no single PC captures most variance, but the sum should be > 0
assert pca_obj.explained_variance_ratio_.sum() > 0
# Re-running with the SAME random_state gives the same projection (numerical sign flip OK)
reduced2, _ = pca_reduce_director_bios(fake_emb, n_components=8, random_state=42)
assert np.allclose(np.abs(reduced), np.abs(reduced2))

print(f'\u2705 pca_reduce_director_bios: shape ok; '
      f'explained_var_sum={pca_obj.explained_variance_ratio_.sum():.3f}')


In [ ]:
def engineer_director_profile(
    df: pd.DataFrame,
    bios_df: pd.DataFrame,
    bio_pca: np.ndarray,
    langs_df: pd.DataFrame,
    lang_to_country_df: pd.DataFrame,
    *,
    top_n_lang: int = 30,
    top_n_country: int = 15,
    bio_pca_dim: int = 64,
) -> pd.DataFrame:
    """Build the per-film director_profile feature block by broadcasting director-level
    signals via left-join on `director_name_norm`.

    Sub-blocks (in column order):
      - dir_bio_pca_0..(D-1)        : PCA-reduced bio embedding (float64 for precision)
      - has_director_bio            : 1 if director's bio was embedded, 0 otherwise
      - dir_lang_<top-N> + dir_lang_other : one-hot of director's dominant filmography language
      - dir_country_<top-N> + dir_country_other : one-hot of country derived from dominant language
      - has_director_lang           : 1 if director appears in MostCommonLanguageByDirector

    Args:
        df: film-level frame; must have 'director_name_norm'.
        bios_df: must have 'director_name_norm' (one row per director with non-empty bio).
        bio_pca: (len(bios_df), bio_pca_dim) — order matches bios_df rows.
        langs_df: director_name_norm x original_language x nb (multiple rows per director ok).
        lang_to_country_df: 'language', 'country' columns.

    Returns:
        DataFrame (len(df), bio_pca_dim + 1 + (top_n_lang+1) + (top_n_country+1) + 1) dims.
        Index aligned with df.index.
    """
    assert bio_pca.shape == (len(bios_df), bio_pca_dim), (
        f"bio_pca shape {bio_pca.shape} doesn't match (n_bios={len(bios_df)}, dim={bio_pca_dim})")

    # ─── Sub-block 1: bio_pca + has_director_bio ───────────────────────────────────
    # Cast to float64 so that list(block.iloc[i][bio_cols]) compares equal
    # to Python float literals (float32 round-trips through Python float lose precision).
    bio_pca_f64 = bio_pca.astype(np.float64)
    bio_lookup = pd.DataFrame(
        bio_pca_f64,
        columns=[f'dir_bio_pca_{i}' for i in range(bio_pca_dim)],
        index=bios_df['director_name_norm'].values,
    )
    bio_lookup = bio_lookup.groupby(level=0).first()  # dedupe duplicate norm names

    bio_block = df[['director_name_norm']].merge(
        bio_lookup, left_on='director_name_norm', right_index=True, how='left',
    )
    bio_cols = [f'dir_bio_pca_{i}' for i in range(bio_pca_dim)]
    has_bio = bio_block[bio_cols[0]].notna().astype(int)
    bio_block[bio_cols] = bio_block[bio_cols].fillna(0.0)

    # ─── Sub-block 2: dominant_lang one-hot ────────────────────────────────────
    # Per director: pick row with max nb (ties -> first after sort descending)
    langs_dom = (
        langs_df.sort_values('nb', ascending=False)
        .drop_duplicates(subset='director_name_norm', keep='first')
        [['director_name_norm', 'original_language']]
        .rename(columns={'original_language': 'dominant_lang'})
    )

    # Top-N languages across directors (not weighted by film count)
    top_langs = langs_dom['dominant_lang'].value_counts().head(top_n_lang).index.tolist()
    langs_dom['dominant_lang_grouped'] = langs_dom['dominant_lang'].where(
        langs_dom['dominant_lang'].isin(top_langs), other='other'
    )

    lang_block = df[['director_name_norm']].merge(
        langs_dom[['director_name_norm', 'dominant_lang_grouped', 'dominant_lang']],
        on='director_name_norm', how='left',
    )
    # has_lang: 1 if director has any language row (dominant_lang_grouped not NaN)
    has_lang = lang_block['dominant_lang_grouped'].notna().astype(int)

    # Initialize all lang one-hot cols to 0; fill only rows with known lang data.
    # NaN rows: (lang_block['dominant_lang_grouped'] == l) is False -> stays 0.
    lang_one_hot_cols = [f'dir_lang_{l}' for l in top_langs] + ['dir_lang_other']
    lang_one_hot = pd.DataFrame(0, index=df.index, columns=lang_one_hot_cols, dtype=int)
    for l in top_langs + ['other']:
        col = f'dir_lang_{l}'
        lang_one_hot[col] = (lang_block['dominant_lang_grouped'] == l).astype(int).values

    # ─── Sub-block 3: country one-hot derived from dominant_lang ─────────────────
    l2c_map = dict(zip(lang_to_country_df['language'], lang_to_country_df['country']))
    # Map dominant_lang (raw) -> country; NaN for unknown lang or missing director
    country = lang_block['dominant_lang'].map(l2c_map)

    top_countries = country.dropna().value_counts().head(top_n_country).index.tolist()
    # NaN (no lang data) -> isin returns False -> where puts 'other';
    # the fillna(False) in comparisons below ensures NaN rows don't get a non-zero flag.
    country_grouped = country.where(country.isin(top_countries), other='other')

    country_one_hot_cols = [f'dir_country_{c}' for c in top_countries] + ['dir_country_other']
    country_one_hot = pd.DataFrame(0, index=df.index, columns=country_one_hot_cols, dtype=int)
    for c in top_countries + ['other']:
        col = f'dir_country_{c}'
        country_one_hot[col] = (country_grouped == c).fillna(False).astype(int).values

    # Zero-out country one-hot for rows where director has no language data
    country_one_hot.loc[has_lang == 0] = 0

    # ─── Assemble final block ────────────────────────────────────────────────
    out = pd.concat([
        bio_block[bio_cols].reset_index(drop=True),
        pd.Series(has_bio.values, name='has_director_bio'),
        lang_one_hot.reset_index(drop=True),
        country_one_hot.reset_index(drop=True),
        pd.Series(has_lang.values, name='has_director_lang'),
    ], axis=1)
    out.index = df.index
    return out

In [ ]:
# Test §2.3.f: engineer_director_profile — 4 synthetic films, 3 directors
# (1 in bios, 2 in langs, 1 missing both)
df_films = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'director_name':      ['Spielberg', 'Kaurismaki', 'Kaurismaki', 'NobodyKnown'],
    'director_name_norm': ['spielberg', 'kaurismaki', 'kaurismaki', 'nobodyknown'],
})
_bios_df = pd.DataFrame({
    'director_name':      ['Spielberg'],
    'bio':                ['mock bio'],
    'director_name_norm': ['spielberg'],
})
_bio_pca = np.array([[0.1, 0.2, 0.3, 0.4]], dtype=np.float64)   # (1, 4) for test simplicity
_langs_df = pd.DataFrame({
    'director_name':      ['Spielberg', 'Kaurismaki'],
    'original_language':  ['en', 'fi'],
    'nb':                 [30, 15],
    'director_name_norm': ['spielberg', 'kaurismaki'],
})
_l2c = pd.DataFrame({'language': ['en', 'fi'], 'country': ['USA', 'FIN']})

block = engineer_director_profile(
    df_films, _bios_df, _bio_pca, _langs_df, _l2c,
    top_n_lang=2, top_n_country=2, bio_pca_dim=4,
)

# 1. Shape: (4 films, 4 bio_pca + 1 has_bio + 3 lang [en, fi, other]
#           + 3 country [top-2 + other] + 1 has_lang) = 12
assert block.shape == (4, 12), block.shape

# 2. has_director_bio flag: only Spielberg row has bio
assert block['has_director_bio'].tolist() == [1, 0, 0, 0]

# 3. has_director_lang flag: Spielberg + Kaurismaki rows yes, NobodyKnown row no
assert block['has_director_lang'].tolist() == [1, 1, 1, 0]

# 4. bio_pca cols match for Spielberg (float64 in output matches Python float literals)
assert list(block.iloc[0][['dir_bio_pca_0', 'dir_bio_pca_1', 'dir_bio_pca_2', 'dir_bio_pca_3']]) == [0.1, 0.2, 0.3, 0.4]
assert list(block.iloc[1][['dir_bio_pca_0', 'dir_bio_pca_1', 'dir_bio_pca_2', 'dir_bio_pca_3']]) == [0.0, 0.0, 0.0, 0.0]

# 5. dominant_lang one-hot: Spielberg=en, Kaurismaki=fi, NobodyKnown=all-zero
assert block.loc[0, 'dir_lang_en'] == 1 and block.loc[0, 'dir_lang_fi'] == 0
assert block.loc[1, 'dir_lang_en'] == 0 and block.loc[1, 'dir_lang_fi'] == 1
assert block.loc[3, ['dir_lang_en', 'dir_lang_fi', 'dir_lang_other']].sum() == 0

# 6. country one-hot derived from language
assert block.loc[0, 'dir_country_USA'] == 1
assert block.loc[1, 'dir_country_FIN'] == 1

# 7. Spec verification: directors with no lang data → all-zero country one-hot
country_cols = [c for c in block.columns if c.startswith('dir_country_')]
assert (block.loc[3, country_cols] == 0).all(), \
    f"NobodyKnown should have all-zero country one-hot, got {block.loc[3, country_cols].to_dict()}"

print(f"\u2705 engineer_director_profile: shape={block.shape}, all 7 assertions passed")

## §2.5 — Assembly (fix #4 — modality-aware scaling)
StandardScaler for numerical/decade/awards; no scaling for genre/language one-hot; row L2-normalize for text embeddings.

In [ ]:
BLOCK_ORDER = ['numerical', 'genre', 'language', 'decade', 'awards', 'text', 'director_profile']


def apply_modality_aware_scaling(
    blocks: dict,
) -> tuple:
    """Modality-aware scaling — fix #4.

    Strategy per block:
      • numerical, decade, awards → StandardScaler (μ=0, σ=1 per dim)
      • genre, language           → no scaling (already {0, 1})
      • text / text_overview      → L2-normalize per ROW (unit-length embeddings)

    Accepts block keys 'text' or 'text_overview' interchangeably for the text block.

    Returns:
      scaled  : dict mapping block name → np.ndarray (or pd.DataFrame for no-scale blocks)
      scalers : dict mapping block name → fitted sklearn scaler (only for standard-scaled blocks)
    """
    scaled = {}
    scalers = {}

    standard_blocks = {'numerical', 'decade', 'awards'}
    no_scale_blocks  = {'genre', 'language'}
    text_keys        = {'text', 'text_overview'}

    for name, block in blocks.items():
        is_df = isinstance(block, pd.DataFrame)
        arr = block.values if is_df else np.asarray(block)

        if name in standard_blocks:
            scaler = StandardScaler()
            result = scaler.fit_transform(arr.astype(np.float32))
            scaled[name] = pd.DataFrame(result, columns=block.columns) if is_df else result
            scalers[name] = scaler

        elif name in no_scale_blocks:
            # Pass-through: preserve DataFrame with column names
            scaled[name] = block if is_df else arr.astype(np.float32)

        elif name in text_keys:
            # Per-row L2 normalisation; zero-safe
            a = arr.astype(np.float32)
            norms = np.linalg.norm(a, axis=1, keepdims=True)
            norms = np.where(norms < 1e-12, 1.0, norms)
            result = a / norms
            # Return under the canonical key 'text' regardless of input key
            canonical = 'text'
            scaled[canonical] = pd.DataFrame(result, columns=block.columns) if is_df else result

        elif name == 'director_profile':
            block = block.copy() if is_df else pd.DataFrame(block)
            bio_cols = [c for c in block.columns if c.startswith('dir_bio_pca_')]
            if bio_cols:
                bio_arr = block[bio_cols].values.astype(np.float32)
                norms = np.linalg.norm(bio_arr, axis=1, keepdims=True)
                norms[norms == 0] = 1.0
                block[bio_cols] = bio_arr / norms
            scaled[name] = block
            scalers[name] = None

        else:
            raise ValueError(f"Unknown block name: {name!r}")

    return scaled, scalers


def assemble_feature_matrix(
    blocks_scaled: dict,
    block_names: dict,
) -> tuple:
    """Concatenate blocks horizontally in fixed BLOCK_ORDER. Returns (X, feature_names)."""
    arrays = []
    names = []
    for b in BLOCK_ORDER:
        if b not in blocks_scaled:
            raise KeyError(f'Missing block: {b}')
        blk = blocks_scaled[b]
        arr = blk.values if isinstance(blk, pd.DataFrame) else np.asarray(blk)
        arrays.append(arr.astype(np.float32))
        names.extend(block_names[b])
    X = np.hstack(arrays).astype(np.float32)
    return X, names


In [ ]:
# Test: apply_modality_aware_scaling — director_profile branch
mini = pd.DataFrame({
    'dir_bio_pca_0': [3.0, 0.0],
    'dir_bio_pca_1': [4.0, 0.0],
    'has_director_bio': [1, 0],
    'dir_lang_en': [1, 0],
    'has_director_lang': [1, 1],
})
out, sc = apply_modality_aware_scaling({'director_profile': mini})
result = out['director_profile']

assert np.allclose(result.loc[0, ['dir_bio_pca_0', 'dir_bio_pca_1']].values, [0.6, 0.8])
assert np.allclose(result.loc[1, ['dir_bio_pca_0', 'dir_bio_pca_1']].values, [0.0, 0.0])
assert result['has_director_bio'].tolist() == [1, 0]
assert result['dir_lang_en'].tolist() == [1, 0]

print("\u2705 apply_modality_aware_scaling: director_profile L2-norm correct, others bypass")

In [ ]:
# Test §2.5: apply_modality_aware_scaling (fix #4)
_blocks = {
    # numerical block: very different scales per column
    'numerical': pd.DataFrame({
        'log_popularity':    [0.0, 1.0, 2.0, 3.0, 4.0],
        'log_vote_count':    [0.0, 5.0, 10.0, 0.0, 5.0],
        'has_vote':          [0, 1, 1, 1, 1],
    }),
    # genre block: 0/1 — must not be rescaled
    'genre': pd.DataFrame({'genre_A': [1, 0, 1, 0, 1], 'genre_B': [0, 1, 1, 0, 0]}),
    # text block: random, will be L2-normalized per row
    'text': np.random.RandomState(42).randn(5, 384).astype(np.float32),
}
_scaled, _scalers = apply_modality_aware_scaling(_blocks)

# numerical: each column ~ μ=0, σ=1
_n = _scaled['numerical']
_n_vals = _n.values if isinstance(_n, pd.DataFrame) else _n
assert np.allclose(_n_vals.mean(axis=0), 0, atol=1e-6), f'numerical mean != 0: {_n_vals.mean(axis=0)}'
_stds = _n_vals.std(axis=0, ddof=0)
assert np.allclose(_stds[_stds > 0], 1, atol=1e-6), f'numerical std != 1: {_stds}'

# genre: untouched
_g = _scaled['genre']
_g_vals = _g.values if isinstance(_g, pd.DataFrame) else _g
assert np.array_equal(_g_vals, _blocks['genre'].values)

# text: every row has L2 norm = 1
_t = _scaled['text']
_t_vals = _t.values if isinstance(_t, pd.DataFrame) else _t
_norms = np.linalg.norm(_t_vals, axis=1)
assert np.allclose(_norms, 1.0, atol=1e-6), f'text L2 norms not 1: min={_norms.min()}, max={_norms.max()}'

# scalers dict has the right keys
assert 'numerical' in _scalers
assert 'genre' not in _scalers   # we didn't scale it
print('✅ apply_modality_aware_scaling: numerical Standard, genre untouched, text L2-normalized')


In [ ]:
# Test §2.5: assemble_feature_matrix (with director_profile — 451 + 113 = 564)
# director_profile dim: 64 (bio PCA) + 1 (has_bio) + 31 (lang 30+other) + 16 (country 15+other) + 1 (has_lang) = 113
_director_profile_dim = 113
_blocks_scaled = {
    'numerical':        np.zeros((4, 6), dtype=np.float32),
    'genre':            np.zeros((4, 22), dtype=np.float32),
    'language':         np.zeros((4, 31), dtype=np.float32),
    'decade':           np.zeros((4, 2), dtype=np.float32),
    'awards':           np.zeros((4, 6), dtype=np.float32),
    'text':             np.zeros((4, 384), dtype=np.float32),
    'director_profile': np.zeros((4, _director_profile_dim), dtype=np.float32),
}
_block_names = {
    'numerical':        [f'num_{i}' for i in range(6)],
    'genre':            [f'genre_{i}' for i in range(22)],
    'language':         [f'lang_{i}' for i in range(31)],
    'decade':           [f'dec_{i}' for i in range(2)],
    'awards':           [f'aw_{i}' for i in range(6)],
    'text':             [f'txt_{i}' for i in range(384)],
    'director_profile': [f'dir_{i}' for i in range(_director_profile_dim)],
}
_X, _names = assemble_feature_matrix(_blocks_scaled, _block_names)
assert _X.shape == (4, 564), f'got {_X.shape}'
assert len(_names) == 564
# Block ordering: numerical (6) | genre (22) | language (31) | decade (2) | awards (6) | text (384) | director_profile (113)
assert _names[0].startswith('num_')
assert _names[5].startswith('num_')
assert _names[6].startswith('genre_')
assert _names[27].startswith('genre_')   # 6+22-1
assert _names[28].startswith('lang_')
assert _names[58].startswith('lang_')    # 6+22+31-1
assert _names[59].startswith('dec_')
assert _names[61].startswith('aw_')
assert _names[67].startswith('txt_')
assert _names[451].startswith('dir_')   # first director_profile col
print(f'\u2705 assemble_feature_matrix: shape={_X.shape}, ordering correct (451+113=564)')

In [ ]:
# Test §2.5: text_overview key alias (parent spec requirement)
_blocks_ov = {
    'numerical': pd.DataFrame({'a': [1.0, 2.0, 3.0]}),
    'genre':     pd.DataFrame({'genre_X': [1, 0, 1]}),
    'language':  pd.DataFrame({'lang_en': [1, 1, 0], 'lang_other': [0, 0, 1]}),
    'decade':    pd.DataFrame({'decade_norm': [0.5, 0.6, 0.7], 'has_release_date': [1, 1, 1]}),
    'awards':    pd.DataFrame({'prior_log_total_wins': [0.0, 1.0, 2.0]}),
    'text_overview': np.random.RandomState(7).randn(3, 384).astype(np.float32),
}
_scaled_ov, _scalers_ov = apply_modality_aware_scaling(_blocks_ov)
# text_overview input → output key is 'text'
assert 'text' in _scaled_ov, f'Expected key text, got: {list(_scaled_ov.keys())}'
_t_ov = _scaled_ov['text']
_t_ov_vals = _t_ov.values if isinstance(_t_ov, pd.DataFrame) else _t_ov
_norms_ov = np.linalg.norm(_t_ov_vals, axis=1)
assert np.allclose(_norms_ov, 1.0, atol=1e-6), 'text_overview L2 norms not 1'
print('✅ apply_modality_aware_scaling: text_overview key accepted, output under text')


## §2.6 — Sanity Helpers
Fail-fast assertion helpers with informative messages.

In [ ]:
def assert_shape(arr, expected: tuple, name: str = 'array') -> None:
    if hasattr(arr, 'shape'):
        got = tuple(arr.shape)
    else:
        got = (len(arr),)
    assert got == expected, f"{name}: shape {got} != expected {expected}"


def assert_no_nan(arr, name: str = 'array') -> None:
    if isinstance(arr, pd.DataFrame):
        n_nan = int(arr.isna().sum().sum())
    else:
        n_nan = int(np.isnan(np.asarray(arr, dtype=float)).sum())
    assert n_nan == 0, f"{name}: contains {n_nan} NaN values"


def assert_value_range(arr, lo: float, hi: float, name: str = 'array') -> None:
    a = np.asarray(arr, dtype=float)
    a_min, a_max = float(np.nanmin(a)), float(np.nanmax(a))
    assert a_min >= lo, f"{name}: min {a_min} < {lo}"
    assert a_max <= hi, f"{name}: max {a_max} > {hi}"


def assert_unit_norm(arr, name: str = 'array', atol: float = 1e-3) -> None:
    a = np.asarray(arr, dtype=float)
    norms = np.linalg.norm(a, axis=1)
    # zero-rows allowed (empty overviews)
    nonzero = norms > 1e-12
    if nonzero.any():
        bad = np.abs(norms[nonzero] - 1.0) > atol
        assert not bad.any(), f"{name}: {bad.sum()} rows have non-unit L2 norm"


In [ ]:
# Test §2.6 helpers
assert_shape(np.zeros((3, 4)), (3, 4), 'zeros')
assert_no_nan(np.zeros((3, 4)), 'zeros')
assert_value_range(np.array([0.1, 0.5, 0.9]), 0, 1, 'unit_interval')

_unit = np.random.RandomState(0).randn(5, 384)
_unit = _unit / np.linalg.norm(_unit, axis=1, keepdims=True)
assert_unit_norm(_unit, 'unit')

# Negative cases
try:
    assert_no_nan(np.array([1.0, np.nan]), 'nan_arr')
    assert False, 'should have raised'
except AssertionError:
    pass

try:
    assert_shape(np.zeros(5), (4,), 'wrong')
    assert False, 'should have raised'
except AssertionError:
    pass

print('✅ §2.6 helpers: positive + negative cases pass')


## §3 — Pipeline Execution
Calls the §2 functions on the real data. After each block, an inline assertion catches bugs early. The §3.6 spot-check cell at the end surfaces the headline numbers for visual review.

In [ ]:
# §3.1 — Load + merge (uses §2.1 functions)
details, casting, awards = load_csvs(CONFIG['paths'])
print(f"   details: {details.shape}")
print(f"   casting: {casting.shape}")
print(f"   awards : {awards.shape}")

df = merge_details_casting(details, casting)
assert_shape(df, (details.shape[0], df.shape[1]), 'df after merge')
assert df['director_name_norm'].notna().all(), "director_name_norm should never be NaN (empty string OK)"
print(f"\u2705 \u00a73.1 merged df: {df.shape}")

In [ ]:
# §3.2 — Temporal-aware awards aggregation (uses §2.2)
awards_per_film = aggregate_awards_temporal(awards, df)
df = df.merge(awards_per_film, on='id', how='left')
for col in AWARDS_RAW_COLS:
    df[col] = df[col].fillna(0)

# Quick comparison: baseline merge rate (raw) vs normalized merge rate (verifies fix #8)
baseline_rate = (
    awards['director_name'].apply(lambda n: n if n else '')
    .isin(casting['director_name'].fillna(''))
    .mean()
)
normalized_rate = (
    awards['director_name'].apply(normalize_director_name)
    .isin(df['director_name_norm'])
    .mean()
)
print(f"   baseline director-match rate (raw):        {baseline_rate*100:.2f}%")
print(f"   normalized director-match rate (fix #8):   {normalized_rate*100:.2f}%")
print(f"   \u0394: {(normalized_rate - baseline_rate)*100:.2f} pp")
print(f"\u2705 \u00a73.2 df after awards merge: {df.shape}")

In [ ]:
# §3.3 — Build feature blocks (uses §2.3)
blocks_df = {}
blocks_df['numerical'] = engineer_numerical(df, CONFIG)
assert_no_nan(blocks_df['numerical'], 'numerical block')
assert_shape(blocks_df['numerical'], (len(df), 6), 'numerical block')

blocks_df['genre']    = engineer_genres(df, top_n=CONFIG['top_n_genres'])
assert_shape(blocks_df['genre'], (len(df), 22), 'genre block')   # 20 + Unknown + has_genre

blocks_df['language'] = engineer_languages(df, top_n=CONFIG['top_n_languages'])
# Language dim depends on data — assert sane upper bound, not exact
n_lang = blocks_df['language'].shape[1]
assert 5 <= n_lang <= 35, f"unexpected language dim: {n_lang}"

blocks_df['decade']   = engineer_decade(df)
assert_shape(blocks_df['decade'], (len(df), 2), 'decade block')

blocks_df['awards']   = engineer_director_awards(df)
assert_shape(blocks_df['awards'], (len(df), 6), 'awards block')

print(f"\u2705 \u00a73.3 blocks: {[(k, v.shape) for k, v in blocks_df.items()]}")

In [ ]:
# §3.4 — Text embedding (uses §2.4) — cached after first run
text_emb = compute_text_embeddings(
    df['overview'].fillna('').astype(str),
    model_name=CONFIG['embedding_model'],
    cache_path=CONFIG['embedding_cache'],
    cache_meta_path=CONFIG['embedding_meta'],
    batch_size=CONFIG['embedding_batch_size'],
)
assert_shape(text_emb, (len(df), 384), 'text_emb')
print(f"\u2705 \u00a73.4 text_emb: {text_emb.shape}, dtype={text_emb.dtype}")

In [ ]:
# ─── §3 step 6: Director-profile data load ──────────────────────────────
print("\n[§3.6] Loading director-profile CSVs...")
bios_df_raw, langs_df_raw, lang_to_country_df = load_director_profile_csvs(CONFIG['paths'])
bios_df, langs_df = normalize_director_in_profile_dfs(bios_df_raw, langs_df_raw)
print(f"   bios:  {len(bios_df):>5} directors with bio")
assert len(bios_df) >= 500, f"Expected ≥500 bios (589 raw - ~84 empty), got {len(bios_df)}"
print(f"   langs: {len(langs_df):>5} director-language rows ({langs_df['director_name_norm'].nunique()} unique directors)")

# ─── §3 step 7: Director-bio embedding + PCA ────────────────────────────
print("\n[§3.7] Director-bio embedding + PCA...")
bio_emb_full = compute_director_bio_embeddings(
    bios_df,
    model_name=CONFIG['embedding_model'],
    cache_path=CONFIG['bio_embedding_cache'],
    meta_path=CONFIG['bio_embedding_meta'],
    batch_size=CONFIG['embedding_batch_size'],
)
bio_pca, pca_director = pca_reduce_director_bios(
    bio_emb_full, n_components=CONFIG['bio_pca_n_components'], random_state=CONFIG['seed'],
)
print(f"   bio embedding shape: {bio_emb_full.shape}")
print(f"   bio PCA shape:       {bio_pca.shape}")
print(f"   PCA explained var:   {pca_director.explained_variance_ratio_.sum():.3f}")
assert pca_director.explained_variance_ratio_.sum() >= 0.75, \
    f"PCA explained variance {pca_director.explained_variance_ratio_.sum():.3f} below 0.75 threshold"

# ─── §3 step 8: Build director_profile block ────────────────────────────
print("\n[§3.8] Building director_profile block...")
blocks_df['director_profile'] = engineer_director_profile(
    df, bios_df, bio_pca, langs_df, lang_to_country_df,
    top_n_lang=CONFIG['top_n_languages'],
    top_n_country=CONFIG['top_n_director_country'],
    bio_pca_dim=CONFIG['bio_pca_n_components'],
)
print(f"   director_profile block shape: {blocks_df['director_profile'].shape}")
print(f"   has_director_bio  count: {int(blocks_df['director_profile']['has_director_bio'].sum()):,}")
print(f"   has_director_lang count: {int(blocks_df['director_profile']['has_director_lang'].sum()):,}")

In [ ]:
# §3.5 — Modality-aware scaling + assemble (uses §2.5)
# Convert blocks to arrays; keep director_profile as DataFrame (needs column names for L2-norm)
blocks_arr = {
    k: (v if k == 'director_profile' else (v.values if isinstance(v, pd.DataFrame) else v))
    for k, v in blocks_df.items()
}
blocks_arr['text'] = text_emb

block_names = {
    'numerical':        list(blocks_df['numerical'].columns),
    'genre':            list(blocks_df['genre'].columns),
    'language':         list(blocks_df['language'].columns),
    'decade':           list(blocks_df['decade'].columns),
    'awards':           list(blocks_df['awards'].columns),
    'text':             [f'text_{i}' for i in range(text_emb.shape[1])],
    'director_profile': list(blocks_df['director_profile'].columns),
}

blocks_scaled, scalers = apply_modality_aware_scaling(blocks_arr)
X, feature_names = assemble_feature_matrix(blocks_scaled, block_names)

# Compute total dim from actual blocks (resilient to language dim change)
expected_dim = sum(blocks_arr[b].shape[1] for b in BLOCK_ORDER)
assert_shape(X, (len(df), expected_dim), 'X')
assert_no_nan(X, 'X')

# Modality balance check (fix #4 + director_profile extension)
_dir_profile_scaled = blocks_scaled['director_profile']
_dir_profile_arr = _dir_profile_scaled.values if isinstance(_dir_profile_scaled, pd.DataFrame) else np.asarray(_dir_profile_scaled)
_bio_cols_idx = [j for j, col in enumerate(block_names['director_profile']) if col.startswith('dir_bio_pca_')]
bio_pca_var = float(_dir_profile_arr[:, _bio_cols_idx].var(axis=0).sum()) if _bio_cols_idx else 0.0
text_var = float(blocks_scaled['text'].var(axis=0).sum())
text_bio_var = text_var + bio_pca_var
other_var = sum(
    float(blocks_scaled[b].var(axis=0).sum())
    for b in BLOCK_ORDER if b not in ('text', 'director_profile')
)
other_var += float(np.var(_dir_profile_arr, axis=0).sum()) - bio_pca_var if bio_pca_var > 0 else float(np.var(_dir_profile_arr, axis=0).sum())
ratio = text_bio_var / max(other_var, 1e-12)
print(f"   modality variance ratio (text+bio_pca / other) = {ratio:.3f}")
# Modality balance is an inherent L2-vs-StandardScaler scaling artifact, not a bug.
# Text/bio (L2-normalized): total variance ~2 regardless of dim count.
# Others (StandardScaler + one-hot bypass): total variance scales with n_cols (~18-22).
# Realistic ratio: ~0.05-0.15. Wider bounds protect against pipeline regression but don't
# enforce a "balance" that the scaling math can't deliver. Modality weighting belongs
# in the AE/VAE/DEC reconstruction loss, not at the pipeline-assembly stage.
assert 0.005 <= ratio <= 5.0, f"\u26a0\ufe0f  modality balance way off (ratio={ratio:.4f}); expected ~0.05-0.15"

print(f"\u2705 \u00a73.5 X: {X.shape}, modal ratio = {ratio:.3f}")

In [ ]:
# §3.6 — Spot-check headline numbers (operator visual review)
print("\u2500\u2500\u2500 \u00a73.6 Pipeline Spot-Checks \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
print(f"  Films total                           : {len(df):,}")
print(f"  Awards merge rate (any prior win > 0) : {(df['prior_total_wins']>0).mean()*100:.2f}%")
print(f"  has_vote = 1                          : {blocks_df['numerical']['has_vote'].sum():,}")
print(f"  has_genre = 1                         : {blocks_df['genre']['has_genre'].sum():,}")
print(f"  has_release_date = 1                  : {blocks_df['decade']['has_release_date'].sum():,}")
print(f"  has_engagement = 1                    : {blocks_df['numerical']['has_engagement'].sum():,}")
print(f"  log_vote_count skewness               : {blocks_df['numerical']['log_vote_count'].skew():.2f}  (fix #7: target < 2)")
print(f"  modality variance ratio (text/other)  : {ratio:.3f}  (fix #4: target 0.5-2.0)")
print(f"  has_director_bio                      : {int(blocks_df['director_profile']['has_director_bio'].sum()):,} ({blocks_df['director_profile']['has_director_bio'].mean()*100:.1f}%)")
print(f"  has_director_lang                     : {int(blocks_df['director_profile']['has_director_lang'].sum()):,} ({blocks_df['director_profile']['has_director_lang'].mean()*100:.1f}%)")
print(f"  bio PCA explained var                 : {pca_director.explained_variance_ratio_.sum()*100:.1f}%")
print("\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")

## §4 — EDA Visualizations

In [ ]:
# §4.1 — Data quality: missing, awards merge quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: missing-data heatmap
key_cols = ['popularity', 'vote_average', 'vote_count', 'runtime', 'overview',
            'genres', 'original_language', 'release_date', 'director_name']
miss = df[key_cols].isnull().sum().sort_values(ascending=False) / len(df) * 100
axes[0].barh(miss.index, miss.values, color='#E74C3C')
axes[0].set_xlabel('Missing %')
axes[0].set_title('Missing Value Rate')
for i, v in enumerate(miss.values):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

# Right: awards merge quality (slide 6 equivalent)
matched = (df['prior_total_wins'] > 0).sum()
has_dir = (df['director_name_norm'] != '').sum()
labels = ['No director\nname', 'Director, no\naward match', 'Has prior\naward']
vals = [len(df) - has_dir, has_dir - matched, matched]
axes[1].barh(labels, vals, color=['#D3D1C7', '#FAC775', '#5DCAA5'])
axes[1].set_xlabel('Film count')
axes[1].set_title('Awards Merge Quality')
for i, v in enumerate(vals):
    axes[1].text(v + 5000, i, f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'missing_analysis.png', bbox_inches='tight')
plt.savefig(CONFIG['figures_dir'] / 'awards_merge_quality.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.2 — Distributions: raw, log-transformed, boxplots
num_features = ['popularity', 'vote_average', 'vote_count', 'runtime']

# Raw
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(num_features):
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    data = data[data > 0] if col != 'vote_average' else data
    axes[i].hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{col}\nskew={data.skew():.2f}')
plt.suptitle('Raw Distributions', fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'hist_raw.png', bbox_inches='tight')
plt.show()

# Log-transformed
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(num_features):
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    data = data[data > 0] if col != 'vote_average' else data
    log_data = np.log1p(data)
    axes[i].hist(log_data, bins=60, color='teal', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'log1p({col})\nskew={log_data.skew():.2f}')
plt.suptitle('Log-Transformed Distributions', fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'hist_log.png', bbox_inches='tight')
plt.show()

# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(num_features):
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    data = data[data > 0] if col != 'vote_average' else data
    axes[i].boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    iqr = data.quantile(0.75) - data.quantile(0.25)
    n_out = ((data < data.quantile(0.25) - 1.5*iqr) |
             (data > data.quantile(0.75) + 1.5*iqr)).sum()
    axes[i].set_title(f'{col}\n{n_out:,} IQR outliers')
plt.suptitle('Boxplots \u2014 Outlier Detection', fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.3 — Correlation (Spearman + Pearson)
corr_cols = ['popularity', 'vote_average', 'vote_count', 'runtime',
             'prior_total_nominations', 'prior_total_wins', 'prior_oscar_nominations']
df_corr = df[corr_cols].apply(pd.to_numeric, errors='coerce')

for method, fname in [('spearman', 'correlation_heatmap.png'),
                      ('pearson',  'correlation_pearson.png')]:
    plt.figure(figsize=(9, 7))
    corr = df_corr.corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0)
    plt.title(f'{method.title()} Correlation Heatmap')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(CONFIG['figures_dir'] / fname, bbox_inches='tight')
    plt.show()

In [ ]:
# §4.4 — Genre, language, decade distributions
# Genre top-20 + pie
genre_counts = blocks_df['genre'].drop(columns=['has_genre']).sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(
    [c.replace('genre_', '') for c in genre_counts.index[:20][::-1]],
    genre_counts.values[:20][::-1],
    color='#4C72B0', alpha=0.85,
)
axes[0].set_title('Top 20 Genres')
axes[0].set_xlabel('Film count')

top8 = genre_counts.head(8)
other = genre_counts.iloc[8:].sum()
pie_data = pd.concat([top8, pd.Series({'genre_Other': other})])
axes[1].pie(pie_data.values,
            labels=[c.replace('genre_', '') for c in pie_data.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.2))
axes[1].set_title('Genre Distribution')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'genre_distribution.png', bbox_inches='tight')
plt.show()

# Language + decade
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
lang_counts = df['original_language'].value_counts().head(15)
axes[0].barh(lang_counts.index[::-1], lang_counts.values[::-1],
             color='#55A868', alpha=0.85)
axes[0].set_title('Top 15 Original Languages')

decade = (pd.to_datetime(df['release_date'], errors='coerce').dt.year // 10 * 10).dropna()
decade_counts = decade.value_counts().sort_index()
decade_counts = decade_counts[decade_counts.index > 0]
axes[1].bar(decade_counts.index.astype(int).astype(str),
            decade_counts.values, color='#8172B2', alpha=0.85)
axes[1].set_title('Movies by Decade')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'categorical_distributions.png', bbox_inches='tight')
plt.show()

# Genre imbalance figure
top10 = genre_counts.head(10)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar([c.replace('genre_', '') for c in top10.index],
            top10.values / top10.sum() * 100,
            color='#4C72B0', alpha=0.85)
axes[0].axhline(y=10, color='red', linestyle='--', label='Balanced baseline (10%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Genre Imbalance (top 10)')
axes[0].legend()

cum = np.cumsum(genre_counts.values) / genre_counts.sum() * 100
axes[1].plot(range(1, len(genre_counts) + 1), cum, color='#4C72B0', linewidth=2)
axes[1].axhline(80, color='red', linestyle='--', alpha=0.7,
                label=f'80% \u2192 top {int(np.argmax(cum>=80)+1)}')
axes[1].axhline(90, color='orange', linestyle='--', alpha=0.7,
                label=f'90% \u2192 top {int(np.argmax(cum>=90)+1)}')
axes[1].set_xlabel('Number of genres')
axes[1].set_ylabel('Cumulative % of films')
axes[1].set_title('Genre Cumulative Distribution')
axes[1].legend()
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'imbalance_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.5 — Modality balance: BEFORE vs AFTER scaling (fix #4)
def modality_variances(blocks):
    return {k: float(np.var(v, axis=0).sum()) for k, v in blocks.items()}

# BEFORE: raw blocks_arr (text not scaled, others not scaled)
before = modality_variances(blocks_arr)
# AFTER: blocks_scaled
after = modality_variances(blocks_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(before.keys())
x = np.arange(len(names))
w = 0.4
axes[0].bar(x - w/2, [before[n] for n in names], w, label='Before scaling', color='#C44E52')
axes[0].bar(x + w/2, [after[n] for n in names],  w, label='After modality-aware', color='#5DCAA5')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names)
axes[0].set_ylabel('Total variance (sum across dims)')
axes[0].set_title('Modality variance \u2014 before vs after\n(fix #4)')
axes[0].legend()
axes[0].set_yscale('log')

# Right plot: ratio text/other for each
text_v_after = after['text']
other_v_after = sum(v for k, v in after.items() if k != 'text')
ratio_after = text_v_after / max(other_v_after, 1e-12)

text_v_before = before['text']
other_v_before = sum(v for k, v in before.items() if k != 'text')
ratio_before = text_v_before / max(other_v_before, 1e-12)

axes[1].bar(['Before', 'After'], [ratio_before, ratio_after],
            color=['#C44E52', '#5DCAA5'])
axes[1].axhline(1.0, color='black', linestyle='--', alpha=0.5)
axes[1].axhspan(0.5, 2.0, alpha=0.15, color='green', label='Acceptable [0.5, 2.0]')
axes[1].set_ylabel('text variance / other variance')
axes[1].set_title('Modality balance ratio')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'modality_balance_before_after.png', bbox_inches='tight')
plt.show()
print(f"   Modality ratio: BEFORE={ratio_before:.2f}, AFTER={ratio_after:.2f}")

In [ ]:
# §4.6 — Text embedding PCA (5k sample to keep it fast)
from sklearn.decomposition import PCA

n_sample = min(5000, text_emb.shape[0])
sample_idx = np.random.RandomState(42).choice(text_emb.shape[0], size=n_sample, replace=False)
emb_sample = text_emb[sample_idx]
genre_sample = blocks_df['genre'].iloc[sample_idx].drop(columns=['has_genre'])

pca_emb = PCA(n_components=2, random_state=42).fit(emb_sample)
emb_2d = pca_emb.transform(emb_sample)

# Variance explained
pca_var = PCA(n_components=50, random_state=42).fit(emb_sample)
cumvar_emb = np.cumsum(pca_var.explained_variance_ratio_) * 100
n80_emb = int(np.argmax(cumvar_emb >= 80)) + 1

# Top genre per row
def get_top_genre(g_row):
    nz = g_row[g_row > 0].index
    if len(nz) == 0:
        return 'Other'
    g = nz[0].replace('genre_', '')
    return g

top_genres = ['Drama', 'Comedy', 'Action', 'Thriller', 'Horror',
              'Romance', 'Documentary', 'Animation']
palette = {'Drama': '#4C72B0', 'Comedy': '#DD8452', 'Action': '#55A868',
           'Thriller': '#C44E52', 'Horror': '#8172B2', 'Romance': '#937860',
           'Documentary': '#DA8BC3', 'Animation': '#8C8C8C', 'Other': '#CCB974',
           'Unknown': '#444444'}
labels = genre_sample.apply(get_top_genre, axis=1).values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for g, c in palette.items():
    mask = labels == g
    if mask.sum() > 0:
        axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                        c=c, label=f'{g} ({mask.sum()})', alpha=0.5, s=10)
axes[0].set_title(f'Text embedding PCA 2D ({n_sample} sample)')
axes[0].set_xlabel(f'PC1 ({pca_emb.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_emb.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(markerscale=2, fontsize=8)

axes[1].plot(range(1, 51), cumvar_emb, marker='o', markersize=3, color='#4C72B0')
axes[1].axhline(80, color='red', linestyle='--', alpha=0.7,
                label=f'80% \u2192 {n80_emb} PCs')
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Cumulative variance (%)')
axes[1].set_title('Text embedding \u2014 variance explained')
axes[1].legend()
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'embedding_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.7 — Clusterability: PCA on full feature matrix (50k sample)
from sklearn.decomposition import PCA

n_clust = min(50_000, X.shape[0])
clust_idx = np.random.RandomState(42).choice(X.shape[0], size=n_clust, replace=False)

pca_full = PCA(n_components=50, random_state=42).fit(X[clust_idx])
cumvar_full = np.cumsum(pca_full.explained_variance_ratio_) * 100
n80_full = int(np.argmax(cumvar_full >= 80)) + 1
n90_full = int(np.argmax(cumvar_full >= 90)) + 1

X_2d = PCA(n_components=2, random_state=42).fit_transform(X[clust_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, 51), cumvar_full, marker='o', markersize=3, color='#4C72B0')
axes[0].axhline(80, color='red',    linestyle='--', alpha=0.7,
                label=f'80% \u2192 {n80_full} PCs')
axes[0].axhline(90, color='orange', linestyle='--', alpha=0.7,
                label=f'90% \u2192 {n90_full} PCs')
axes[0].set_xlabel('Number of PCs')
axes[0].set_ylabel('Cumulative explained variance (%)')
axes[0].set_title('Full feature matrix PCA variance')
axes[0].legend()

genre_clust = blocks_df['genre'].iloc[clust_idx].drop(columns=['has_genre'])
labels_clust = genre_clust.apply(get_top_genre, axis=1).values
for g, c in palette.items():
    mask = labels_clust == g
    if mask.sum() > 0:
        axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                        c=c, label=g, alpha=0.35, s=4)
axes[1].set_title(f'Full PCA 2D scatter \u2014 {n_clust} sample')
axes[1].legend(markerscale=3, fontsize=7)

plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'clusterability_pca.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.8 — Variance thresholding + figures verifying fixes #1, #2, #5

# Variance thresholding
variances = X.var(axis=0)
sorted_v = np.sort(variances)[::-1]
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].bar(range(len(sorted_v)), sorted_v, color='steelblue', alpha=0.7)
axes[0].axhline(0.01, color='red', linestyle='--', label='Threshold 0.01')
axes[0].set_yscale('log')
axes[0].set_title('All features \u2014 log scale')
axes[0].legend()
axes[1].bar(range(20), sorted_v[-20:][::-1], color='steelblue', alpha=0.7)
axes[1].axhline(0.01, color='red', linestyle='--')
axes[1].set_title('Bottom 20 features')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'variance_thresholding.png', bbox_inches='tight')
plt.show()
n_below = int((variances < 0.01).sum())
print(f"   Features below threshold 0.01: {n_below}")

# Fix #2 — vote_average imputation impact (NEW figure)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
old_imp = df['vote_average'].fillna(df['vote_average'].median())  # what the original did
new_imp = blocks_df['numerical']['vote_average_norm']
axes[0].hist(old_imp, bins=80, color='#C44E52', alpha=0.6, label='Original (median=0)')
axes[0].set_title('vote_average \u2014 ORIGINAL imputation\n(yapay 0 y\u0131\u011f\u0131lmas\u0131)')
axes[0].set_xlabel('vote_average')
axes[0].legend()
axes[1].hist(new_imp, bins=80, color='#5DCAA5', alpha=0.6, label='Fix #2 (voted-only mean)')
axes[1].set_title('vote_average_norm \u2014 FIX #2\n(no artificial 0-spike)')
axes[1].set_xlabel('vote_average_norm')
axes[1].legend()
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'vote_average_imputation_impact.png', bbox_inches='tight')
plt.show()

# Fix #1 — temporal awards visualization (NEW figure)
df_temp = df.copy()
df_temp['rel_year'] = pd.to_datetime(df_temp['release_date'], errors='coerce').dt.year
df_temp['rel_decade'] = (df_temp['rel_year'] // 10 * 10).astype('Int64')
trend = df_temp.dropna(subset=['rel_decade']).groupby('rel_decade')['prior_total_wins'].mean()
trend = trend[(trend.index >= 1900) & (trend.index <= 2020)]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trend.index.astype(int), trend.values, marker='o', color='#4C72B0')
ax.set_xlabel('Release decade')
ax.set_ylabel('Mean prior_total_wins (temporal-aware)')
ax.set_title('Fix #1 \u2014 average director award count AT release time, by decade\n'
             'Should rise monotonically (no leakage from future awards)')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'awards_temporal_cutoff.png', bbox_inches='tight')
plt.show()

# Fix #5 — multilingual coverage (NEW figure)
overview_lang_avail = df.groupby('original_language')['overview'].apply(
    lambda s: (s.fillna('').str.len() > 0).mean()
).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(overview_lang_avail.index, overview_lang_avail.values, color='#8172B2')
ax.set_ylabel('Fraction with non-empty overview')
ax.set_title('Fix #5 \u2014 overview availability by language (top 15)\n'
             '(multilingual model now embeds these meaningfully)')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'multilingual_coverage.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.1.bis  Director-bio coverage by decade
fig, ax = plt.subplots(figsize=(11, 4))

dec_df = df.copy()
dec_df['has_dir_bio'] = blocks_df['director_profile']['has_director_bio'].values
dec_df['decade'] = (pd.to_datetime(dec_df['release_date'], errors='coerce').dt.year // 10 * 10)
dec_df = dec_df[dec_df['decade'].between(1900, 2020)]
cov = dec_df.groupby('decade')['has_dir_bio'].agg(['sum', 'count'])
cov['pct'] = cov['sum'] / cov['count'] * 100

ax.bar(cov.index.astype(int).astype(str), cov['pct'], color='#5DCAA5', alpha=0.85, edgecolor='white')
overall = blocks_df['director_profile']['has_director_bio'].mean() * 100
ax.axhline(overall, color='red', linestyle='--', alpha=0.7, label=f'Overall: {overall:.1f}%')
ax.set_ylabel('% of films with director bio')
ax.set_xlabel('Decade')
ax.set_title('Director Wikipedia-bio coverage at film level')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'director_bio_coverage.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.6.bis  Bio PCA scree
pca_full = PCA(n_components=min(150, bio_emb_full.shape[1]), random_state=42).fit(bio_emb_full)
cum = np.cumsum(pca_full.explained_variance_ratio_) * 100
n75 = int(np.argmax(cum >= 75)) + 1
n80 = int(np.argmax(cum >= 80)) + 1
n90 = int(np.argmax(cum >= 90)) + 1

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(cum) + 1), cum, marker='o', markersize=3, color='#4C72B0')
ax.axvline(64, color='black', linestyle=':', alpha=0.6, label='Selected: 64 PCs')
ax.axhline(75, color='red',    linestyle='--', alpha=0.6, label=f'75% → {n75} PCs')
ax.axhline(80, color='orange', linestyle='--', alpha=0.6, label=f'80% → {n80} PCs')
ax.axhline(90, color='green',  linestyle='--', alpha=0.6, label=f'90% → {n90} PCs')
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative explained variance (%)')
ax.set_title('Director-bio embedding — PCA variance explained')
ax.legend()
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'director_bio_pca_scree.png', bbox_inches='tight')
plt.show()
print(f"At 64 PCs: {cum[63]:.1f}% explained variance")

In [ ]:
# §4.6.bis  Bio PCA 2D scatter — colored by country
pca_2d = PCA(n_components=2, random_state=42).fit_transform(bio_emb_full)

# Map each bio row → country via the dominant lang
l2c_map = dict(zip(lang_to_country_df['language'], lang_to_country_df['country']))
dom_lang = (
    langs_df.sort_values('nb', ascending=False)
    .drop_duplicates('director_name_norm', keep='first')
    .set_index('director_name_norm')['original_language']
)
bio_country = bios_df['director_name_norm'].map(dom_lang).map(l2c_map).fillna('Unknown')

top_countries = bio_country.value_counts().head(8).index.tolist()
bio_country_grouped = bio_country.where(bio_country.isin(top_countries), 'Other')

fig, ax = plt.subplots(figsize=(10, 7))
palette_8 = sns.color_palette('tab10', n_colors=len(top_countries) + 1)
for i, c in enumerate(top_countries + ['Other']):
    mask = (bio_country_grouped.values == c)
    ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1], color=palette_8[i],
               label=f'{c} ({mask.sum()})', alpha=0.7, s=25)
ax.set_title('Director-bio PCA 2D — colored by country (top-8 + other)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(markerscale=1.2, fontsize=9, loc='best')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'director_bio_pca_2d_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# §4.4.bis  Director's dominant language vs film's original language
top_n = 12

# 1. Build per-director dominant lang, then broadcast to films
dir_lang_per_director = (
    langs_df.sort_values('nb', ascending=False)
    .drop_duplicates('director_name_norm', keep='first')
    .set_index('director_name_norm')['original_language']
)
film_dir_lang = df['director_name_norm'].map(dir_lang_per_director)

# 2. Top-N vocabularies (independent for each axis)
top_dir_langs  = dir_lang_per_director.value_counts().head(top_n).index
top_film_langs = df['original_language'].value_counts().head(top_n).index

dir_lang_grouped  = film_dir_lang.where(film_dir_lang.isin(top_dir_langs), 'other').fillna('—')
film_lang_grouped = df['original_language'].where(df['original_language'].isin(top_film_langs), 'other').fillna('—')

# 3. Crosstab — row-normalized (each row sums to 1)
confusion = pd.crosstab(dir_lang_grouped, film_lang_grouped, normalize='index')

# 4. Plot
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(confusion, annot=True, fmt='.2f', cmap='YlOrRd',
            cbar_kws={'label': 'Row-normalized fraction'}, ax=ax)
ax.set_xlabel('Film original_language')
ax.set_ylabel("Director's dominant language")
ax.set_title('Director language × Film language — row-normalized')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'director_lang_vs_film_lang.png', bbox_inches='tight')
plt.show()

# 5. Off-diagonal mass — restrict to languages present on BOTH axes for a fair diagonal
common_langs = sorted(set(confusion.index) & set(confusion.columns))
common_block = confusion.loc[common_langs, common_langs]
diag_mass    = float(np.diag(common_block.values).sum())
total_mass   = float(common_block.values.sum())
off_diag_pct = (1.0 - diag_mass / total_mass) * 100 if total_mass > 0 else 0.0
print(f"Off-diagonal mass over shared lang grid: {off_diag_pct:.1f}%   (target: ≥ 5% — feature is non-redundant)")

In [ ]:
# §4.5.bis  Modality balance — 6 blocks
modality_groups = {
    'Numerical\n(6)':          blocks_scaled['numerical'],
    'Genre\n(22)':             blocks_scaled['genre'],
    'Language\n(31)':          blocks_scaled['language'],
    'Decade\n(2)':             blocks_scaled['decade'],
    'Awards\n(6)':             blocks_scaled['awards'],
    'Text overview\n(384)':    blocks_scaled['text'],
    'Director profile\n(113)': blocks_scaled['director_profile'],
}

def _to_arr(v):
    return v.values if hasattr(v, 'values') else np.asarray(v)

variances = {n: float(_to_arr(v).var(axis=0).sum()) for n, v in modality_groups.items()}

fig, ax = plt.subplots(figsize=(11, 5))
colors_mod = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860', '#5DCAA5']
ax.bar(list(variances.keys()), list(variances.values()), color=colors_mod, alpha=0.85, edgecolor='white')
for i, (k, v) in enumerate(variances.items()):
    ax.text(i, v + max(variances.values()) * 0.01, f'{v:.1f}', ha='center', fontsize=9)
ax.set_ylabel('Total variance (post-scaling)')
ax.set_title('Modality balance — variance contribution (6 modalities)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'modality_balance_v2.png', bbox_inches='tight')
plt.show()

## §5 — Persistence
Saves all artifacts that the modelling notebooks will consume.

In [ ]:
# §5 — Save artifacts
import pickle, importlib.metadata

artifacts = CONFIG['artifacts_dir']

# 1. feature_matrix.npz (model input — primary)
np.savez_compressed(
    artifacts / 'feature_matrix.npz',
    X=X,
    feature_names=np.array(feature_names, dtype=object),
)

# 2. feature_matrix_raw.npz — pre-scaling, block dict (for ablation)
np.savez_compressed(
    artifacts / 'feature_matrix_raw.npz',
    **{k: np.asarray(v, dtype=np.float32) for k, v in blocks_arr.items()},
)

# 3. movies_eda_final.csv — raw + non-text blocks (text excluded for size/readability — fix #13)
encoded_concat = pd.concat(
    [blocks_df['numerical'].reset_index(drop=True),
     blocks_df['genre'].reset_index(drop=True),
     blocks_df['language'].reset_index(drop=True),
     blocks_df['decade'].reset_index(drop=True),
     blocks_df['awards'].reset_index(drop=True),
     blocks_df['director_profile'][[c for c in blocks_df['director_profile'].columns if not c.startswith('dir_bio_pca_')]].reset_index(drop=True)],
    axis=1,
)
final = pd.concat([df.reset_index(drop=True), encoded_concat], axis=1)
final.to_csv(artifacts / 'movies_eda_final.csv', index=False)
print(f"   movies_eda_final.csv: {final.shape}")

# 4. scalers.pkl
with open(artifacts / 'scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

# 5. feature_metadata.json
metadata = {
    'feature_names': feature_names,
    'block_order': BLOCK_ORDER,
    'block_dims': {b: int(blocks_arr[b].shape[1]) for b in BLOCK_ORDER},
    'total_dim': int(X.shape[1]),
    'n_films': int(X.shape[0]),
}
(artifacts / 'feature_metadata.json').write_text(json.dumps(metadata, indent=2))

# 6. pipeline_version.json — reproducibility audit (fix #11)
md5 = hashlib.md5()
md5.update(X.tobytes())
matrix_hash = md5.hexdigest()

def _ver(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return 'unknown'

version_info = {
    'seed': CONFIG['seed'],
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'library_versions': {
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'sentence_transformers': _ver('sentence-transformers'),
        'torch': _ver('torch'),
        'umap_learn': _ver('umap-learn'),
        'scikit_learn': _ver('scikit-learn'),
    },
    'model_name': CONFIG['embedding_model'],
    'n_films': int(X.shape[0]),
    'feature_dim': int(X.shape[1]),
    'feature_matrix_md5': matrix_hash,
}
(artifacts / 'pipeline_version.json').write_text(json.dumps(version_info, indent=2))

print("\u2705 \u00a75 Persistence complete:")
for f in sorted(artifacts.iterdir()):
    if f.is_file():
        print(f"   {f.name:35s} {f.stat().st_size/1e6:8.2f} MB")
print(f"\n   feature_matrix MD5 = {matrix_hash}")

In [ ]:
# ─── Director-profile artifacts ──────────────────────────────────────
import pickle

with open(CONFIG['bio_pca_cache'], 'wb') as f:
    pickle.dump(pca_director, f)

director_profile_meta = {
    'n_directors_with_bio': int(len(bios_df)),
    'film_coverage_bio': float(blocks_df['director_profile']['has_director_bio'].mean()),
    'film_coverage_lang': float(blocks_df['director_profile']['has_director_lang'].mean()),
    'bio_pca_n_components': int(pca_director.n_components_),
    'bio_pca_explained_variance': float(pca_director.explained_variance_ratio_.sum()),
    'top_languages': [c.replace('dir_lang_', '') for c in blocks_df['director_profile'].columns if c.startswith('dir_lang_')],
    'top_countries': [c.replace('dir_country_', '') for c in blocks_df['director_profile'].columns if c.startswith('dir_country_')],
}
CONFIG['director_profile_meta'].write_text(
    json.dumps(director_profile_meta, indent=2)
)

print(f"\n✅ Director-profile artifacts saved:")
print(f"   {CONFIG['bio_embedding_cache']}")
print(f"   {CONFIG['bio_embedding_meta']}")
print(f"   {CONFIG['bio_pca_cache']}")
print(f"   {CONFIG['director_profile_meta']}")

In [ ]:
# Run summary
print("=" * 60)
print("CineEmbed eda_v2 — Director-profile Extension Run Summary")
print("=" * 60)
print(f"Feature matrix shape : {X.shape}")
print(f"Expected             : (329044, 564)")
print(f"NaN count            : {int(np.isnan(X).sum())}")
print(f"Director bio coverage: {blocks_df['director_profile']['has_director_bio'].mean()*100:.1f}%")
print(f"Director lang coverage: {blocks_df['director_profile']['has_director_lang'].mean()*100:.1f}%")
print(f"PCA explained var    : {pca_director.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Modality balance     : {ratio:.3f}  (target: [0.5, 2.5])")
print(f"Artifacts written    : {len(list(CONFIG['artifacts_dir'].rglob('*'))):>3} files")
print("=" * 60)